In [ ]:
!pip install -q transformers datasets evaluate accelerate sentencepiece scikit-learn

import os
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

from datasets import Dataset
import evaluate
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

base_dir = "/content/drive/MyDrive/CME"
data_dir = os.path.join(base_dir, "data")

merged_path = os.path.join(data_dir, "merged_all.csv")
clothing_path = os.path.join(data_dir, "clothing_reviews.csv")

print("Merged dataset path:", merged_path)
print("Clothing dataset path:", clothing_path)


In [ ]:
merged_df = pd.read_csv(merged_path)

print("Merged_all columns:", merged_df.columns)
print("Merged_all shape:", merged_df.shape)


required_cols = ["review_text", "label", "rating", "language", "domain"]
for col in required_cols:
    if col not in merged_df.columns:
        raise ValueError(f"Column '{col}' is missing from merged_all.csv")


if merged_df["label"].dtype != int and merged_df["label"].dtype != "int64":
    print("Label is not int; please map it manually here if needed.")

merged_df["rating"] = merged_df["rating"].fillna(-1)

print(merged_df.head())


In [ ]:
train_df, test_df = train_test_split(
    merged_df,
    test_size=0.2,
    random_state=42,
    stratify=merged_df["label"]
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

In [ ]:
clothing_df = pd.read_csv(clothing_path)

print("Clothing dataset columns:", clothing_df.columns)
print("Clothing dataset shape:", clothing_df.shape)

for col in required_cols:
    if col not in clothing_df.columns:
        raise ValueError(f"Column '{col}' is missing from clothing_reviews.csv")


if clothing_df["label"].dtype != int and clothing_df["label"].dtype != "int64":
    print("Clothing labels are not int; add mapping here if needed.")

clothing_df["rating"] = clothing_df["rating"].fillna(-1)

print(clothing_df.head())

In [ ]:
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["review_text"],
        padding="max_length",
        truncation=True,
        max_length=256,
    )


In [ ]:
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)


train_ds = train_ds.remove_columns(["review_text"])
test_ds = test_ds.remove_columns(["review_text"])


train_ds.set_format("torch")
test_ds.set_format("torch")

print(train_ds)
print(test_ds)

In [ ]:
import evaluate
import numpy as np


accuracy_metric  = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric    = evaluate.load("recall")
f1_metric        = evaluate.load("f1")

def compute_metrics(eval_pred):

    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc  = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    prec = precision_metric.compute(predictions=preds, references=labels, average="binary")["precision"]
    rec  = recall_metric.compute(predictions=preds, references=labels, average="binary")["recall"]
    f1   = f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"]

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
    }

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
)
model.to(device)


training_args = TrainingArguments(
    output_dir=os.path.join(base_dir, "model_output"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
save_dir = os.path.join(base_dir, "model_final")
os.makedirs(save_dir, exist_ok=True)

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print("Model saved to:", save_dir)

In [ ]:
reload_dir = os.path.join(base_dir, "model_final")

tokenizer = AutoTokenizer.from_pretrained(reload_dir)
model = AutoModelForSequenceClassification.from_pretrained(reload_dir)
model.to(device)

print("Reloaded model from:", reload_dir)

In [ ]:
internal_results = trainer.evaluate(test_ds)
print("=== Internal 20% Test Set (mixed domains, EN+AR) ===")
print(internal_results)

preds_output = trainer.predict(test_ds)
y_true_internal = preds_output.label_ids
y_pred_internal = preds_output.predictions.argmax(-1)

print("\nClassification report (internal):")
print(classification_report(y_true_internal, y_pred_internal, digits=2))

print("Confusion matrix (internal):")
print(confusion_matrix(y_true_internal, y_pred_internal))

In [ ]:
clothing_ds = Dataset.from_pandas(clothing_df.reset_index(drop=True))
clothing_ds = clothing_ds.map(tokenize, batched=True)
clothing_ds = clothing_ds.remove_columns(["review_text"])
clothing_ds.set_format("torch")

preds_clothing = trainer.predict(clothing_ds)
y_true_clothing = preds_clothing.label_ids
y_pred_clothing = preds_clothing.predictions.argmax(-1)

print("\n=== Zero-shot Clothing Domain Test Set ===")
print("Classification report (clothing):")
print(classification_report(y_true_clothing, y_pred_clothing, digits=2))

print("Confusion matrix (clothing):")
print(confusion_matrix(y_true_clothing, y_pred_clothing))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

reload_dir = "/content/drive/MyDrive/CME/model_final"

tokenizer = AutoTokenizer.from_pretrained(
    reload_dir,
    local_files_only=True,
    use_fast=False
)

model = AutoModelForSequenceClassification.from_pretrained(
    reload_dir,
    local_files_only=True
)

model.to(device)
model.eval()

print("Model loaded successfully!")

In [ ]:
inputs = tokenizer("test", return_tensors="pt").to(device)
outputs = model(**inputs)
outputs.logits

In [ ]:
import torch
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def clean_tokens(tokens):
    cleaned = []
    for t in tokens:
        if t in ["<s>", "</s>"]:
            continue
        if all(c in ".,!?;:" for c in t):
            continue
        cleaned.append(t)
    return cleaned

def explain_attention(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True).to(device)

    outputs = model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        output_attentions=True
    )

    attentions = outputs.attentions
    last_layer = attentions[-1][0]


    cls_attn = last_layer[:, 0, :]
    cls_attn = cls_attn.mean(dim=0).detach().cpu().numpy()

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    tokens = clean_tokens(tokens)

    cls_attn = cls_attn[:len(tokens)]

    plt.figure(figsize=(14, 4))
    sns.barplot(x=tokens, y=cls_attn, palette="Blues")
    plt.xticks(rotation=70)
    plt.title("Attention Weights (CLS → Tokens)")
    plt.ylabel("Attention Score")
    plt.xlabel("Token")
    plt.show()

example_text = "I have had mine for two years now. Very reliable and extremely useful."
explain_attention(example_text)

In [ ]:
import pandas as pd

new_df = pd.read_csv("/content/drive/MyDrive/CME/data/clothing_reviews.csv")

In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader
import numpy as np


def tokenize_batch(batch):
    return tokenizer(batch["review_text"], truncation=True, padding=True, return_tensors="pt")


reviews = new_df["review_text"].tolist()
true_labels = new_df["label"].tolist()

pred_labels = []

model.eval()
with torch.no_grad():
    for text in reviews:
        enc = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
        outputs = model(**enc)
        pred = torch.argmax(outputs.logits, dim=-1).cpu().item()
        pred_labels.append(pred)

new_df["predicted_label"] = pred_labels

def get_case(row):
    if row["label"] == 1 and row["predicted_label"] == 1:
        return "TP"
    if row["label"] == 0 and row["predicted_label"] == 0:
        return "TN"
    if row["label"] == 0 and row["predicted_label"] == 1:
        return "FP"
    if row["label"] == 1 and row["predicted_label"] == 0:
        return "FN"

new_df["case"] = new_df.apply(get_case, axis=1)

display(new_df[["review_text", "label", "predicted_label", "case"]].head(20))

print("\n=== Confusion Summary ===")
print("TP:", sum(new_df["case"] == "TP"))
print("TN:", sum(new_df["case"] == "TN"))
print("FP:", sum(new_df["case"] == "FP"))
print("FN:", sum(new_df["case"] == "FN"))